In [1]:
from typing import List, Tuple, Union, Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import contractions
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
import nltk
from nltk.downloader import download
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, recall_score, precision_recall_curve, roc_curve, auc, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

/home/jeffmoe/CST_645_Week5/venv/lib/python3.10/site-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
download('popular')

[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading package inaugural to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package inaugural is already up-to-date!
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package movie_reviews is already

True

In [3]:
def prepare_data_from_csv(
    file_path: str,
    text_column: str,
    label_column: str,
):
    """
    Loads CSV, preprocesses text, and encodes labels to 0/1.

    Args:
        file_path: path to CSV file
        text_column: column name containing text
        label_column: column name containing labels
        positive_label: value representing positive sentiment

    Returns:
        tokens_list: list of tokenized documents
        labels: list of 0/1 labels
    """
    # creating progress bar
    tqdm.pandas()

    # using pandas to load in the csv file of comments
    df = pd.read_csv(file_path)

    # remove missing values from the dataset
    df = df[[text_column,label_column]].dropna()

    # init the stop words and lemmatizer
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    def preprocess(text):
        try:
            # Handle input types
            if isinstance(text, list):
                text = ' '.join(text)
            elif not isinstance(text, str):
                return []

            # expand contractions
            text = contractions.fix(text)

            # tokenize
            tokens = nltk.word_tokenize(text)

            # normalize and clean
            tokens = [
                re.sub(r'[^a-zA-Z0-9]', '', w.lower())
                for w in tokens
            ]

            # remove empties and stop words
            tokens = [w for w in tokens if w and w not in stop_words]

            # lemmatize the words
            tokens = [lemmatizer.lemmatize(w) for w in tokens]

            return tokens

        except Exception as e:
            print(f"Error preprocessing text: {e}")
            return []

    # apply the text preprocessing
    print("Preprocessing text data...")
    tokens_list = df[text_column].progress_apply(preprocess).tolist()

    # encode labels for analysis in the SVM pipeline
    unique_labels = sorted(df[label_column].astype(str).str.lower().unique())
    if len(unique_labels) != 2:
        raise ValueError(
            f"Expected binary labels, but found: {unique_labels}"
        )
    label_mapping = {
        unique_labels[0]: 0,
        unique_labels[1]: 1
    }
    print(f"Label mapping: {label_mapping}")
    labels = df[label_column].astype(str).str.lower().map(label_mapping).tolist()

    return tokens_list, labels, label_mapping


In [4]:
def svm_pipeline(tokens: Union[List[List[str]], Tuple[List[str]]],
                 labels: List[int]) -> Dict[str, Any]:
    """
    Builds a TF-IDF + Linear SVM pipeline with GridSearchCV.

    Args:
        tokens: list of tokenized documents (each doc = list of words)
        labels: list of binary labels (1 = positive, 0 = negative)

    Returns:
        dict containing:
            - best_model
            - best_params
            - X_train, X_test, y_train, y_test
            - y_pred
            - decision_scores
            - classification_report
            - confusion_matrix
            - predict_fn (for new tokens)
    """

    # Join tokens into strings for splitting data
    print("Preparing text data...")
    text_data = [' '.join(t) for t in tqdm(tokens, desc="Joining Tokens")]

    # splitting our data into training and testing sets for the model
    print("Splitting data...")
    X_train, X_test, y_train, y_test = train_test_split(
        text_data,
        labels,
        test_size=0.2,
        random_state=42,
        stratify=labels
    )

    # Creating a repeatable pipeline for model retraining using sklearn
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('svm', LinearSVC(random_state=42, dual=False))
    ])

    # Using a hyperparameter grid to determine the best values for the model
    parameter_grid = {
        'tfidf__max_df': [0.75, 0.85, 1.0],
        'tfidf__min_df': [1, 2, 5],
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'tfidf__sublinear_tf': [True, False],
        'svm__C': [0.01, 0.1, 1.0, 10.0]
    }
    total_fits = (
        len(parameter_grid["tfidf__max_df"]) *
        len(parameter_grid["tfidf__min_df"]) *
        len(parameter_grid["tfidf__ngram_range"]) *
        len(parameter_grid["tfidf__sublinear_tf"]) *
        len(parameter_grid["svm__C"]) *
        5
    )



    # setting up our Grid search to implement cross-validation and determing the hyperparameters
    print(f"Running our grid search (~{total_fits} fits)...")
    gridsearch = GridSearchCV(
        pipeline,
        parameter_grid,
        cv=5,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    with tqdm_joblib(tqdm(desc="Grid Search Progress", total=total_fits)):
        # model training
        gridsearch.fit(X_train, y_train)
    print("Grid search is now done.")

    best_model = gridsearch.best_estimator_

    # using the best model from the grid search for our predictions
    print("Getting predictions...")
    y_pred = best_model.predict(X_test)

    # decision function is a good built in for visualizing hyperplanes with scatter plots
    decision_scores = best_model.decision_function(X_test)

    # getting the accuracy metrics for the model
    report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)

    # function when wanting to test new data to see what the predicted outcome would be
    def predict_new(new_tokens: List[List[str]]):
        new_text = [' '.join(t) for t in new_tokens]
        preds = best_model.predict(new_text)
        scores = best_model.decision_function(new_text)
        return preds, scores

    return {
        "best_model": best_model,
        "best_params": gridsearch.best_params_,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "y_pred": y_pred,
        "decision_scores": decision_scores,
        "classification_report": report,
        "confusion_matrix": cm,
        "predict_fn": predict_new
    }

In [5]:
def analyze_svm_results(results: dict):
    """
    Generates evaluation visualizations and metrics
    from svm_pipeline output.

    Args:
        results: dictionary returned from svm_pipeline()
    """

    y_test = results["y_test"]
    y_pred = results["y_pred"]
    scores = results["decision_scores"]

    # creating the scatterplot based on the decision function
    plt.figure()
    plt.scatter(scores, y_test, alpha=0.5)
    plt.xlabel("Decision Score")
    plt.ylabel("True Label")
    plt.title("Decision Scores vs True Labels")
    plt.grid(True)
    plt.show()

    # creating the precision recall curve 
    precision, recall, _ = precision_recall_curve(y_test, scores)

    plt.figure()
    plt.plot(recall, precision)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.grid(True)
    plt.show()

    # creating the roc curve (receiver operating characteristic) and area under the curve (auc) for analysis
    # receiver operating characteristic: True pos. rate (TPR) vs the false pos rate (FPR)
    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True)
    plt.show()

    # using seaborn to display the confusion matrix from the model
    cm = confusion_matrix(y_test, y_pred)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()

    # displaying the classification report of the model
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))


In [ ]:
tokens, labels, mapping = prepare_data_from_csv(
    file_path='IMDB Dataset.csv',
    text_column='review',
    label_column='sentiment'
)
print(mapping)
results = svm_pipeline(tokens,labels)
analyze_svm_results(results)

Preprocessing text data...


100%|██████████| 50000/50000 [01:37<00:00, 510.34it/s]


Label mapping: {'negative': 0, 'positive': 1}
{'negative': 0, 'positive': 1}
Preparing text data...


Joining Tokens: 100%|██████████| 50000/50000 [00:00<00:00, 206195.24it/s]


Splitting data...
Running our grid search (~720 fits)...


Grid Search Progress:   0%|          | 0/720 [00:00<?, ?it/s]

  0%|          | 0/720 [00:00<?, ?it/s]